In [ ]:
import pandas as pd 
import holidays

In [19]:
# Charger le jeu de données
df = pd.read_csv('./dataset/champs_elysees.csv', sep=";")

# Renommer les colonnes pour un accès plus facile (pas d'espaces, pas d'accents)
df.rename(columns={
    'Date et heure de comptage': 'datetime',
    'Débit horaire': 'debit_horaire',
    "Taux d'occupation": 'taux_occupation',
    'Libelle noeud amont': 'noeud_amont',
    'Libelle noeud aval': 'noeud_aval'
}, inplace=True)

# Conversion de la colonne 'datetime' en objet datetime de pandas
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

# Définir la colonne 'datetime' comme index du DataFrame
df.set_index('datetime', inplace=True)

# Trier les données par date, crucial pour une série temporelle correcte
df.sort_index(inplace=True)

In [20]:
# Créer la feature pour les jours fériés
fr_holidays = holidays.France()
print(fr_holidays)

{'country': FR, 'expand': True, 'language': None, 'market': None, 'observed': True, 'subdiv': None, 'years': set()}


In [22]:
# --- Création des features de calendrier (votre code original, maintenant fonctionnel) ---

df['est_ferie'] = df.index.to_series().apply(lambda date: date.date() in fr_holidays).astype(int)

# Créer la feature pour les vacances scolaires (Toussaint 2025)
df['est_vacances_scolaires'] = ((df.index.date >= pd.to_datetime('2025-10-18').date()) & (df.index.date <= pd.to_datetime('2025-11-03').date())).astype(int)

# La veille des jours fériés
df['veille_ferie'] = df['est_ferie'].shift(-1).fillna(0).astype(int)


# --- Vérification des features créées ---

print("="*50)
print("Affichage des jours fériés trouvés dans le dataset")
print("="*50)
# Filtrer le DataFrame pour ne montrer que les jours où 'est_ferie' est 1
jours_feries_df = df[df['est_ferie'] == 1]
print(jours_feries_df[['debit_horaire', 'taux_occupation', 'est_ferie']])


print("\n" + "="*50)
print("Affichage du début de la période de vacances scolaires")
print("="*50)
vacances_df = df[df['est_vacances_scolaires'] == 1]
print(vacances_df[['debit_horaire', 'taux_occupation', 'est_vacances_scolaires']])


print("\n" + "="*50)
print("Affichage des veilles de jours fériés")
print("="*50)
veilles_feries_df = df[df['veille_ferie'] == 1]
print(veilles_feries_df[['debit_horaire', 'taux_occupation', 'veille_ferie', 'est_ferie']])

Affichage des jours fériés trouvés dans le dataset
                           debit_horaire  taux_occupation  est_ferie
datetime                                                            
2024-11-01 00:00:00+00:00            NaN              NaN          1
2024-11-01 01:00:00+00:00            NaN              NaN          1
2024-11-01 02:00:00+00:00            NaN              NaN          1
2024-11-01 03:00:00+00:00            NaN              NaN          1
2024-11-01 04:00:00+00:00            NaN              NaN          1
...                                  ...              ...        ...
2025-08-15 19:00:00+00:00          841.0         20.59778          1
2025-08-15 20:00:00+00:00          843.0         29.61167          1
2025-08-15 21:00:00+00:00          841.0         17.73833          1
2025-08-15 22:00:00+00:00          740.0         16.60945          1
2025-08-15 23:00:00+00:00          873.0         16.47444          1

[241 rows x 3 columns]

Affichage du début de la pé